# Analyse results from Pypsa-Earth
This notebook reads the latest optimization described in the config.yaml file and makes plots and summaries of the results.

Sources: 
- Plot capacity - map view: https://github.com/pypsa-meets-earth/documentation/blob/main/notebooks/viz/regional_transm_system_viz.ipynb
- Analyse energy potential: https://github.com/pypsa-meets-earth/documentation/blob/main/notebooks/build_renewable_profiles.ipynb
- Analyse energy generation: https://pypsa.readthedocs.io/en/latest/examples/statistics.html

Some files are needed:
* PyPSA network file (e.g. "elec.nc" contains a lot of details and looks perfect)
* a country shape file (may be found in "resources/shapes/country_shapes.geojson")
* a renewable profile file (may be found in "resources/renewable_profiles/....nc)

## Import packages

In [ ]:
import yaml
import pypsa
import warnings
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from datetime import datetime
from cartopy import crs as ccrs
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches
import os
import xarray as xr
import cartopy

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)
# change current directory to parent folder
if not os.path.isdir("pypsa-earth"):
    os.chdir("../..")

PARENT = os.path.realpath("pypsa-earth/") + "/"
# Specify config name
CONFIG = "config-EG-sec-2050-0_50_100_200_300_400_800_1000TWh"
config = yaml.safe_load(open(PARENT + "own-configs/" + CONFIG + ".yaml"))

# Specify name for figures
overall_name = 'EG_400TWh'
# Specify of you want to save the figures
save_figs = False

In [ ]:
run_name = config["run"]["name"]
run_sector_name = config["run"]["sector_name"]                          
simpl = config["scenario"]["simpl"]    
clust = config["scenario"]["clusters"]   
ll = config["scenario"]["ll"]             
load_scale = config["load_options"]["scale"]               
opts = config["scenario"]["opts"]       
sopts = config["scenario"]["sopts"]              
planning = config["scenario"]["planning_horizons"]        
discountrate = config["costs"]["discountrate"]                
demand = config["scenario"]["demand"]                  
# export_value = config["export"]["h2export"]  
export_value = [400]                

# Process each setting into a string representation 
simpl_str = "_".join(map(str, simpl))   
clust_str = "_".join(map(str, clust))    
ll_str = "l" + "_".join(map(str, ll))           
scale_str = f"lc{load_scale}"                           
opts_str = "_".join(map(str, opts))    
sopts_str = "_".join(map(str, sopts))                 
planning_str = "_".join(map(str, planning))               
dr_str = "_".join(map(str, discountrate))             
demand_str = "_".join(map(str, demand))                    
export_str = "_".join(map(str, export_value)) + "export"

scenario_subpath = f"{run_name}/" if run_name else ""

nc_file_name = (
    f"elec_s_{clust_str}_ec_{ll_str}_{opts_str}_{sopts_str}_"
    f"{planning_str}_{dr_str}_{demand_str}_{export_str}.nc"
)

# Country shape file
regions_onshore_path = PARENT + f"resources/{scenario_subpath}shapes/country_shapes.geojson"

# Bus_regions_file
bus_regions = PARENT + f"resources/{scenario_subpath}bus_regions/regions_onshore.geojson"

# Results path
results_path = PARENT + f"results/{run_sector_name}/postnetworks/{nc_file_name}"

# Renewable profile file
solar_path = PARENT + f"resources/{scenario_subpath}renewable_profiles/profile_solar.nc"
onwind_path = PARENT + f"resources/{scenario_subpath}renewable_profiles/profile_onwind.nc"

## Analysis Network Setup

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)
n = pypsa.Network(results_path)
regions_onshore = gpd.read_file(regions_onshore_path)
bus_regions = gpd.read_file(bus_regions)
country_coordinates = regions_onshore.total_bounds[[0, 2, 1, 3]]
warnings.simplefilter(action='default', category=FutureWarning)

## Data import check

Plot of the region of interest

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": ccrs.EqualEarth(n.buses.x.mean())})
# with plt.rc_context({"patch.linewidth": 0.}):
#     regions_onshore.plot(
#     ax=ax,
#     facecolor="green",
#     edgecolor="white",
#     aspect="equal",
#     transform=ccrs.PlateCarree(),
#     linewidth=0,
#     )
# ax.set_title(", ".join(regions_onshore.name.values))

List number of components by type

In [ ]:
for c in n.iterate_components(list(n.components.keys())[2:]):
    print("Component '{}' has {} entries".format(c.name,len(c.df)))

List the snapshots of the PyPSA network

In [ ]:
print(n.snapshots)
print(f"Time steps: " + str(len(n.snapshots)))

## Analyse energy system

In [ ]:
n.carriers.loc[n.carriers.color.isnull() | (n.carriers.color == "#11111"), "color"] = "#d3d3d3"
n.carriers.color

Analyse the current capacity of the energy system - map view

In [ ]:
missing_carriers = set(buses.index.get_level_values(1)) - set(n.carriers.index)

for carrier in missing_carriers:
    n.carriers.loc[carrier, "color"] = "#d3d3d3"  # Assign a default color (light gray)

print(missing_carriers)

In [ ]:
gen_no_ac = gen[~gen.index.get_level_values(0).str.contains("AC")]
gen_no_ac = gen_no_ac.drop("Earth lignite", errors="ignore")
gen_no_ac

In [ ]:
# Set default color for carriers with missing or invalid color values
n.carriers.loc[n.carriers.color.isnull() | (n.carriers.color == "#11111") | (n.carriers.color == ""), "color"] = "#d3d3d3"
n.carriers.color

### Specify custom bounds

In [ ]:
bounds = regions_onshore.total_bounds.copy()  # [minx, miny, maxx, maxy]

# Add padding in degrees (e.g., 2 degrees on each side)
pad_x = 2
pad_y = 2

#bounds[0] -= pad_x  # expand west
bounds[2] += pad_x  # expand east
#bounds[1] -= pad_y  # expand south
#bounds[3] += pad_y  # expand north

print(f"Old bounds: {regions_onshore.total_bounds}\nNew bounds: {bounds}")

In [ ]:
# Scale settings
bus_scale = 3e5 
line_scale = 6e3
link_scale = 6e25

bus_sizes = [1000, 10000, 50000]  # in MW
line_sizes = [1000, 10000, 50000]  # in MW

# # Legend settings
# bus_sizes = [100, 1000]  # in MW
# line_sizes = [100, 1000]  # in MW

#n.carriers.drop("load", inplace=True)
fig, ax = plt.subplots(figsize=(20, 8), subplot_kw={"projection": ccrs.EqualEarth(n.buses[n.buses.carrier == 'AC'].x.mean())})
gen = n.generators[n.generators.carrier != "load"].groupby(["bus", "carrier"]).p_nom.sum()
gen = gen[gen > 100]
gen.drop("Earth lignite", errors="ignore")
gen = gen[gen != 0]
sto = n.storage_units.groupby(["bus", "carrier"]).p_nom.sum()
buses = pd.concat([gen, sto])

with plt.rc_context({"patch.linewidth": 0.}):
    n.plot(
        bus_sizes=buses / bus_scale,
        bus_alpha=0.7,
        line_widths=n.lines.s_nom / line_scale,
        link_widths=n.links[n.links.carrier == 'AC'].p_nom_opt / link_scale, # filter for AC in order to show nothing (hotfix)
        # link_widths=n.links.p_nom_opt / link_scale,
        line_colors="#7ba6db",
        ax=ax,
        margin=0.4,
        color_geomap={'ocean': 'white', 'land': 'whitesmoke'},
    )
# regions_onshore.plot(
#     ax=ax,
#     facecolor="whitesmoke",
#     edgecolor="white",
#     aspect="equal",
#     transform=ccrs.PlateCarree(),
#     linewidth=0,
# )
# ax.set_extent(regions_onshore.total_bounds[[0, 2, 1, 3]])
ax.set_extent(bounds[[0, 2, 1, 3]])
legend_kwargs = {"loc": "upper left", "frameon": False}
legend_circles_dict = {"bbox_to_anchor": (1.1, 0.67), "labelspacing": 2.5, **legend_kwargs}

add_legend_circles(
    ax,
    [s / bus_scale for s in bus_sizes],
    [f"{s / 1000} GW" for s in bus_sizes],
    patch_kw={'facecolor':'#7ba6db'},
    legend_kw=legend_circles_dict,    
)
add_legend_lines(
    ax,
    [s / line_scale for s in line_sizes],
    [f"{s / 1000} GW" for s in line_sizes],
    colors=["#7ba6db"] * len(line_sizes),
    legend_kw={"bbox_to_anchor": (1.1, 0.8), **legend_kwargs},
)
add_legend_patches(
    ax,
    n.carriers.iloc[0:10,:].color,
    n.carriers.iloc[0:10,:].nice_name,
    legend_kw={"bbox_to_anchor": (1.1, 0), **legend_kwargs, "loc":"lower left"},
)
fig.tight_layout()

# Ensure the directory exists
output_dir = f"pypsa-earth-docu/results/EG/graphics_general/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    filename = f"{output_dir}{overall_name}_brownfield_network.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    filename_png = filename.replace('.pdf', '.png')
    plt.savefig(filename_png, bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
(n.lines.s_nom_opt - n.lines.s_nom).sort_values(ascending=False).head(10)

Analyse the current generation capacity of the energy system - pie chart view

In [ ]:
generator_capacity_by_carrier = (
    n.generators[n.generators.carrier != "load"]
    .groupby(["carrier"])
    .p_nom.sum()
)
plt.figure(figsize=(8, 8))
plt.pie(
    generator_capacity_by_carrier,
    labels=generator_capacity_by_carrier.index,
    autopct='%1.1f%%',
    colors=n.carriers[
            n.carriers.index.isin(generator_capacity_by_carrier.index)
        ].color.reindex(generator_capacity_by_carrier.index)
)
plt.title("Generator Capacity by Carrier")

Analyse the current gernation capacity of the energy system - tabular view

In [ ]:
generator_capacity_by_carrier/1e3 # in GW

Analyse the future capacity of the energy system - map view

In [ ]:
gen[gen > 100].sort_values(ascending=False).plot(kind='bar', figsize=(10, 6))

## Updated: Plot optimized network 

In [ ]:
geomap_custom = regions_onshore
geomap_custom = geomap_custom.to_crs("EPSG:4326")

### Specify custom bounds

In [ ]:
bounds = regions_onshore.total_bounds.copy()  # [minx, miny, maxx, maxy]

# Add padding in degrees (e.g., 2 degrees on each side)
pad_x = 2
pad_y = 2

#bounds[0] -= pad_x  # expand west
bounds[2] += pad_x  # expand east
#bounds[1] -= pad_y  # expand south
#bounds[3] += pad_y  # expand north

print(f"Old bounds: {regions_onshore.total_bounds}\nNew bounds: {bounds}")

In [ ]:
n.links[n.links.carrier == 'H2 pipeline repurposed'].p_nom_opt.sort_values(ascending=False).head()

In [ ]:
# Scale settings
bus_scale = 3e5 
line_scale = 6e3
link_scale = 6e25

bus_sizes = [1000, 10000, 50000]  # in MW
line_sizes = [1000, 10000, 50000]  # in MW

fig, ax = plt.subplots(figsize=(20, 8), subplot_kw={"projection": ccrs.EqualEarth(n.buses[n.buses.carrier == 'AC'].x.mean())})
gen = n.generators[n.generators.carrier != "load"].groupby(["bus", "carrier"]).p_nom_opt.sum()
gen = gen[gen > 100]
gen.drop("Earth lignite", errors="ignore")
gen = gen[gen != 0]
sto = n.storage_units.groupby(["bus", "carrier"]).p_nom_opt.sum()
buses = pd.concat([gen, sto])

# geomap_custom.plot(ax=ax, facecolor="whitesmoke", edgecolor="white", aspect="equal", transform=ccrs.PlateCarree(), linewidth=0)

with plt.rc_context({"patch.linewidth": 0.}):
    n.plot(
        bus_sizes=buses / bus_scale,
        bus_alpha=0.7,
        line_widths=n.lines.s_nom_opt / line_scale,
        link_widths=n.links[n.links.carrier == 'AC'].p_nom_opt / link_scale, # filter for AC in order to show nothing (hotfix)
        # link_widths=n.links[n.links.carrier == 'AC'].p_nom_opt / link_scale,
        geomap=True,
        geometry=None,
        line_colors="#7ba6db",
        ax=ax,
        margin=0.4,
        color_geomap={'ocean': 'white', 'land': 'whitesmoke'},
    )
# regions_onshore.plot(
#     ax=ax,
#     facecolor="whitesmoke",
#     edgecolor="white",
#     aspect="equal",
#     transform=ccrs.PlateCarree(),
#     linewidth=0,
# )

# bus_regions.plot(
#     ax=ax,
#     facecolor="whitesmoke",
#     edgecolor="black",
#     linewidth=1,
#     aspect="equal",
#     transform=ccrs.PlateCarree()
# )

#ax.set_extent(regions_onshore.total_bounds[[0, 2, 1, 3]])
ax.set_extent(bounds[[0, 2, 1, 3]])
legend_kwargs = {"loc": "upper left", "frameon": False}
legend_circles_dict = {"bbox_to_anchor": (1.1, 0.67), "labelspacing": 2.5, **legend_kwargs}

add_legend_circles(
    ax,
    [s / bus_scale for s in bus_sizes],
    [f"{s / 1000} GW" for s in bus_sizes],
    patch_kw={'facecolor':'#7ba6db'},
    legend_kw=legend_circles_dict,    
)
add_legend_lines(
    ax,
    [s / line_scale for s in line_sizes],
    [f"{s / 1000} GW" for s in line_sizes],
    colors=["#7ba6db"] * len(line_sizes),
    legend_kw={"bbox_to_anchor": (1.1, 0.8), **legend_kwargs},
)
add_legend_patches(
    ax,
    n.carriers.iloc[0:10,:].color,
    n.carriers.iloc[0:10,:].nice_name,
    legend_kw={"bbox_to_anchor": (1.1, 0), **legend_kwargs, "loc":"lower left"},
)
fig.tight_layout()

# Ensure the directory exists
output_dir = f"pypsa-earth-docu/results/EG/graphics_general/"
os.makedirs(output_dir, exist_ok=True)

# Save figure without white borders
if save_figs:
    filename = f"{output_dir}{overall_name}_optimized_network.pdf"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    filename_png = filename.replace('.pdf', '.png')
    plt.savefig(filename_png, bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
bounds = regions_onshore.total_bounds.copy()  # [minx, miny, maxx, maxy]

# Add padding in degrees (e.g., 2 degrees on each side)
pad_x = 2
pad_y = 2

#bounds[0] -= pad_x  # expand west
bounds[2] += pad_x  # expand east
#bounds[1] -= pad_y  # expand south
#bounds[3] += pad_y  # expand north

print(f"Old bounds: {regions_onshore.total_bounds}\nNew bounds: {bounds}")

## Analyse loads (updated)

In [ ]:
n.loads.carrier.unique()

In [ ]:
loads_sum = (
    n.loads_t.p
    .groupby([n.loads.bus, n.loads.carrier], axis=1)
    .sum()
    .mul(n.snapshot_weightings.generators.mean())
    .sum()
)
loads_sum[loads_sum.abs() > 1e6].divide(1e6).sort_values(ascending=False).plot(kind='bar', figsize=(10, 6))

In [ ]:
n.statistics.energy_balance(comps='Load').divide(-1e6).sort_values(ascending=False).plot(kind='bar', figsize=(10, 6))

In [ ]:
# Fix: Use get_level_values to filter MultiIndex by 'heat' in any level
energy_balance_load = n.statistics.energy_balance(comps='Load')
mask = energy_balance_load.index.to_frame().apply(lambda row: row.astype(str).str.contains('heat').any(), axis=1)
energy_balance_load[mask].divide(-1e6).sort_values(ascending=False).plot(kind='bar', figsize=(10, 6))

In [ ]:
# Fix: Use get_level_values to filter MultiIndex by 'transport' in any level
energy_balance_load = n.statistics.energy_balance(comps='Load')
mask = energy_balance_load.index.to_frame().apply(lambda row: row.astype(str).str.contains('transport').any(), axis=1)
energy_balance_load[mask].divide(-1e6).sort_values(ascending=False).plot(kind='bar', figsize=(10, 6))

Analys the future generation capacity expansion of the energy system - bar chart

In [ ]:
optimal_capacity = n.statistics.optimal_capacity(comps=["Generator"]).droplevel(0).div(1e3)
installed_capacity = n.statistics.installed_capacity(comps=["Generator"]).droplevel(0).div(1e3)
generation_capacity_expansion = optimal_capacity - installed_capacity
generation_capacity_expansion.drop(["load"], inplace=True)
generation_capacity_expansion.plot.bar(title="Generator capacity expansion in GW")

Plot the future generation capacity expansion of the energy system - tabular chart

In [ ]:
generation_capacity_expansion # In GW

Analyse the future energy generation of the energy system - bar chart view

In [ ]:
colors = {key.lower(): value.lower() for key, value in config["plotting"]["tech_colors"].items()}
nice_names = {value.lower(): key for key, value in config["plotting"]["nice_names"].items()}

rename_cols = {
    '-': 'Load',
    'load': 'load shedding',
}

energy_balance = (
    n.statistics.energy_balance()
    .loc[:, :, "AC"]
    .groupby("carrier")
    .sum()
    .div(1e6)
    .to_frame()
    .T
    .rename(columns=rename_cols)
)

# color-matching
color_list = []
for col in energy_balance.columns:
    original_name = col.lower()
    key_name = nice_names.get(original_name, original_name)
    color = colors.get(key_name.lower(), 'gray')
    color_list.append(color)


fig, ax = plt.subplots()
energy_balance.plot.bar(stacked=True, ax=ax, title="Energy Balance in TWh", color=color_list)
handles, labels = ax.get_legend_handles_labels()
nice_labels = [config["plotting"]["nice_names"].get(label, label) for label in energy_balance.columns]
ax.legend(handles, nice_labels, bbox_to_anchor=(1, 0), loc="lower left", title=None, ncol=1)

plt.show()

Analyse the future energy generation of the energy system - tabular view

In [ ]:
n.statistics.energy_balance()/1e6 # In TWh

## Analyse pv and wind potential - map view

In [ ]:
solar = xr.open_dataset(solar_path)
wind = xr.open_dataset(onwind_path)

def plot_voronoi(n, carrier, voronoi, cmap, projection, title=None, filename=None):
    g = n.generators.loc[n.generators.carrier == carrier]
    br = gpd.read_file(f"{PARENT}resources/{scenario_name}/bus_regions/regions_{voronoi}.geojson").set_index("name")
    br_area = br.to_crs("ESRI:54009")
    br_area = br_area.geometry.area * 1e-6
    br["p_nom_max"] = g.groupby("bus").sum().p_nom_max / br_area

    fig, ax = plt.subplots(figsize=(8, 4), subplot_kw={"projection": projection})
    plt.rcParams.update({"font.size": 10})
    br.plot(
        ax=ax,
        column="p_nom_max",
        transform=ccrs.PlateCarree(),
        linewidth=0.25,
        edgecolor="k",
        cmap=cmap,
        vmin=0,
        vmax=br["p_nom_max"].max(),
        legend=True,
        legend_kwds={"label": r"potential density"},
    )
    ax.coastlines()
    ax.add_feature(cartopy.feature.BORDERS.with_scale("110m"))
    ax.set_extent(country_coordinates, crs=ccrs.PlateCarree()) 
    
    if title is not None:
        plt.title(title)

Plot wind energy potential

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)
plot_voronoi(
    pypsa.Network(network_path),
    "onwind",
    "onshore",
    "Blues",
    ccrs.PlateCarree(),
    title="Onshore Wind Potential Density [MW/km2]",
)
warnings.simplefilter(action='default', category=FutureWarning)

Plot pv energy potential

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)
plot_voronoi(
    pypsa.Network(network_path),
    "solar",
    "onshore",
    "OrRd",
    ccrs.PlateCarree(),
    title="Solar Photovoltaic Potential Density [MW/km2]",
)
warnings.simplefilter(action='default', category=FutureWarning)